# TabPFN Regression — DIMER E2E tutorial

[GitHub](https://github.com/kurtvalcorza/tabpfn-regressor-pipeline) · [Open in Colab](https://colab.research.google.com/github/kurtvalcorza/tabpfn-regressor-pipeline/blob/main/tutorials/tabpfn_regressor_colab.ipynb) · [Model](https://huggingface.co/Prior-Labs/TabPFN-v2-reg) · [Upstream](https://github.com/PriorLabs/TabPFN) · [Model card](https://github.com/kurtvalcorza/tabpfn-regressor-pipeline/blob/main/MODEL_CARD.md)

**Profile:** `E2E` · **DIMER Notebook Specification:** 1.0

This notebook runs supervised tabular **regression** through the repository's production code path: the DIMER fine-tuner worker (`train.run()` from `tabpfn-regressor-finetuner` at the commit pinned in `COMPONENTS.json`) and the serving loader (`serving/load_artifact.py`). TabPFN is a prior-data fitted network: in the default mode `fit()` registers the labelled training rows as in-context support and performs **no gradient update**; with `FINE_TUNE=True` and a CUDA GPU the worker runs Prior Labs' gradient fine-tuning wrapper. Without a usable GPU a fine-tune request does **not** fail — the worker falls back to zero-shot ICL and records `fineTuneEffective=false` with `fineTuneSkippedReason`, which this notebook surfaces rather than hides. The upstream `tabpfn==8.1.0` package supplies the model and weights; this repository adds the DIMER contract, dataset validation, artifact packaging, the reload check, and the serving loader.

By the end, you will pin and verify the worker revision, build or upload a dataset, run the worker exactly as DIMER does, read its metrics against a trivial baseline, reload the exported artifact from serialized files through the serving loader, prove the reloaded estimator reproduces the recorded validation metrics, score new rows, and export CSV/JSON results and provenance.

**Boundaries.** Regression only; no classification and no per-prediction uncertainty — `predict()` returns a raw point estimate (negative values are valid), with no interval. The default generation is `v2` because its weights carry Prior Labs' Apache-derived licence; `v3` (default in DIMER) is selectable here but its weights are under `tabpfn-3-license-v1.0`, whose Non-Commercial Purpose excludes production deployment. Sample metrics on synthetic data are tutorial sanity evidence only, not benchmark or production evidence.

**Prerequisites.** Clean Google Colab runtime with Python 3.11+; CPU is sufficient for the default zero-shot path, a CUDA GPU is required for effective fine-tuning (Prior Labs recommends 80 GB for the full wrapper; the tutorial's small sample fits a T4). Internet access is required for the two repositories and the model weights. BYOD stays in the runtime; do not upload restricted data to an unauthorized environment.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys
if sys.version_info < (3, 11):
    raise RuntimeError("This tutorial requires Python 3.11+.")
PIPE_DIR = Path("/content/tabpfn-regressor-pipeline")
FT_DIR = Path("/content/tabpfn-regressor-finetuner")
for d in (PIPE_DIR, FT_DIR):
    if d.exists():
        shutil.rmtree(d)
!git clone -q https://github.com/kurtvalcorza/tabpfn-regressor-pipeline.git /content/tabpfn-regressor-pipeline
COMPONENTS = json.loads((PIPE_DIR / "COMPONENTS.json").read_text(encoding="utf-8"))
FT_REPO = COMPONENTS["components"]["finetuner"]["repository"]
FT_COMMIT = COMPONENTS["components"]["finetuner"]["commit"]
!git clone -q https://github.com/{FT_REPO}.git /content/tabpfn-regressor-finetuner
!git -C /content/tabpfn-regressor-finetuner checkout -q {FT_COMMIT}
%pip install -q -r /content/tabpfn-regressor-pipeline/tutorials/requirements-colab.txt
!git -C /content/tabpfn-regressor-pipeline rev-parse HEAD

## 1. Runtime identity and immutable worker revision

The worker code is pinned by the immutable commit recorded in `COMPONENTS.json`; the next cell verifies the clone actually sits at that commit and prints the effective Python/torch/tabpfn versions and device. TabPFN weights are resolved by the `tabpfn` package for the selected generation; the worker records the resolved base-model path and, when DIMER supplies a checkpoint, its SHA-256 — this tutorial uses the package download, so the record names the generation rather than a file digest.

In [ ]:
import platform, importlib.metadata as md
import numpy as np, pandas as pd, torch
head = subprocess.check_output(["git", "-C", str(FT_DIR), "rev-parse", "HEAD"], text=True).strip()
if head != FT_COMMIT:
    raise RuntimeError(f"finetuner checkout {head} != pinned {FT_COMMIT}")
print("Python", platform.python_version(), "torch", md.version("torch"), "tabpfn", md.version("tabpfn"),
      "pandas", md.version("pandas"), "sklearn", md.version("scikit-learn"))
print("Device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print({"finetunerRepository": FT_REPO, "finetunerCommit": FT_COMMIT, "pipelineRepository": "kurtvalcorza/tabpfn-regressor-pipeline"})
sys.path.insert(0, str(FT_DIR))
sys.path.insert(0, str(PIPE_DIR))
import train as worker
from serving.load_artifact import load_dimer_tabpfn_artifact
IS_CLASSIFIER = False

## 2. Sample or BYOD dataset

Default data is a synthetic linear-plus-category table (600 rows: `x1`, `x2`, `category`) from `examples/build_synthetic_dataset.py` — synthetic, deterministic from the seed, and used only to prove the workflow, not as benchmark evidence. It is packaged as a ZIP with explicit `train.csv`, `val.csv`, and `test.csv`, which the worker preserves rather than re-splitting. For BYOD set `USE_BYOD=True` and upload one ZIP with the same layout (or a single `train.csv`, in which case the worker draws a seeded holdout of `VALIDATION_SPLIT`): each CSV needs unique column names and a finite numeric `target` column. Random splitting assumes rows are independent; for temporal, grouped, or patient-level data supply your own leakage-safe `val.csv`/`test.csv`. The worker enforces the selected generation's row/feature caps and its own archive-safety limits (per-file size, expanded size, compression ratio, path safety) before any model work.

In [ ]:
USE_BYOD = False  # @param {type:"boolean"}
TARGET_COLUMN = "target"  # @param {type:"string"}
VALIDATION_SPLIT = 0.2  # @param {type:"number"}
DATASET_DIR = Path("/content/dimer/dataset")
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one dataset ZIP or train.csv.")
    name, raw = next(iter(uploaded.items()))
    if not name.lower().endswith((".zip", ".csv")):
        raise ValueError("BYOD must be a ZIP (train/val/test CSVs) or a single train.csv.")
    (DATASET_DIR / ("train.csv" if name.lower().endswith(".csv") else name)).write_bytes(raw)
    data_source = f"user upload: {name}"
else:
    subprocess.run([sys.executable, str(PIPE_DIR / "examples" / "build_synthetic_dataset.py"),
                    "--out", str(DATASET_DIR / "synthetic.zip"), *"--rows 600 --seed 42".split()], check=True)
    data_source = "synthetic sample from examples/build_synthetic_dataset.py (--rows 600 --seed 42)"
import hashlib, zipfile
zips = sorted(DATASET_DIR.glob("*.zip"))
dataset_digest = hashlib.sha256(zips[0].read_bytes()).hexdigest() if zips else hashlib.sha256((DATASET_DIR / "train.csv").read_bytes()).hexdigest()
def _read(name):
    if zips:
        with zipfile.ZipFile(zips[0]) as z:
            members = {Path(n).name: n for n in z.namelist() if n.endswith(".csv")}
            return pd.read_csv(z.open(members[name])) if name in members else None
    p = DATASET_DIR / name
    return pd.read_csv(p) if p.exists() else None
train_df, val_df, test_df = _read("train.csv"), _read("val.csv"), _read("test.csv")
if train_df is None or TARGET_COLUMN not in train_df.columns or train_df.columns.duplicated().any():
    raise ValueError("train.csv with unique columns and the target column is required.")
print(data_source, "| sha256", dataset_digest[:16])
print("train", train_df.shape, "val", None if val_df is None else val_df.shape, "test", None if test_df is None else test_df.shape)
print("features:", [c for c in train_df.columns if c != TARGET_COLUMN])
print("target summary:", train_df[TARGET_COLUMN].value_counts().sort_index().to_dict() if IS_CLASSIFIER else train_df[TARGET_COLUMN].describe().round(3).to_dict())

## 3. Run the DIMER worker

The worker is configured exactly as DIMER configures it: through `DIMER_*` environment variables and the two JSON hyperparameter/preprocessing transports. `MODEL_VERSION` selects the TabPFN generation (`v2` default here for licence reasons; `v2.5`, `v2.6`, `v3` are the DIMER options). `FINE_TUNE=False` runs zero-shot in-context learning and is the CPU-capable default; `FINE_TUNE=True` requests gradient fine-tuning, which is effective only on a CUDA device. `train.run()` validates the dataset, fits, evaluates on `val.csv` (and `test.csv` when present), saves `model.tabpfn_fit` + `model.ckpt` + `artifact_manifest.json`, and performs its own reload check before writing `result.json`.

**What to look for:** `successful: true`, `fineTuneEffective`, and — if you requested fine-tuning without a GPU — the `fineTuneSkippedReason` explaining the zero-shot fallback. `mae` is average absolute error in target units; `rmse` weights large errors more; `r2` is relative to a constant-mean reference and can be negative; `mapepercent` is computed only over rows whose true value is non-zero (`maperows`). These are single-split tutorial numbers with no dispersion estimate.

In [ ]:
import os
MODEL_VERSION = "v2"  # @param ["v2", "v2.5", "v2.6", "v3"]
FINE_TUNE = False  # @param {type:"boolean"}
EPOCHS = 3  # @param {type:"integer"}
SEED = 42  # @param {type:"integer"}
OUTPUT_DIR, RESULT_PATH = Path("/content/dimer/output"), Path("/content/dimer/results/result.json")
for d in (OUTPUT_DIR, RESULT_PATH.parent):
    if d.exists():
        shutil.rmtree(d)
os.environ.update({
    "DIMER_DATASET_DIR": str(DATASET_DIR), "DIMER_OUTPUT_DIR": str(OUTPUT_DIR), "DIMER_RESULT_PATH": str(RESULT_PATH),
    "DIMER_TRAIN_DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    "DIMER_EXPECTED_ACCELERATOR": "gpu" if torch.cuda.is_available() else "cpu",
    "DIMER_PREPROCESSING_ARGS_JSON": json.dumps({"target_column": TARGET_COLUMN, "validation_split": VALIDATION_SPLIT}),
    "DIMER_HYPERPARAMETERS_JSON": json.dumps({"model_version": MODEL_VERSION, "fine_tune": FINE_TUNE, "epochs": EPOCHS, "seed": SEED,
                                              "n_estimators_finetune": 2, "n_estimators_validation": 2, "n_estimators_final_inference": 4}),
    "DIMER_PIPELINE_METADATA_JSON": json.dumps({"taskType": "tabular_regression"}),
    "DIMER_RUN_ID": "colab-tutorial", "DIMER_SESSION_ID": "colab-tutorial", "DIMER_DONE_CALLBACK": "",
})
if FINE_TUNE and not torch.cuda.is_available():
    print("WARNING: FINE_TUNE=True without CUDA — the worker will run zero-shot ICL and record fineTuneSkippedReason.")
config = worker.load_config()
rc = worker.run(config)
result = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
if rc != 0 or not result.get("successful"):
    raise RuntimeError(f"worker failed: {result.get('message')}")
metrics = result["metrics"]
print("worker:", result["message"], "| mode:", metrics.get("mode"), "| fineTuneEffective:", metrics.get("fineTuneEffective"))
if metrics.get("fineTuneSkippedReason"):
    print("fineTuneSkippedReason:", metrics["fineTuneSkippedReason"])
print("validation:", metrics["validation"])
if metrics.get("test"):
    print("test:", metrics["test"])

## 4. Trivial baseline

The training-mean constant predictor is the trivial baseline, scored on the same `val.csv` the worker evaluated (and `test.csv` when present) so the comparison uses identical rows. The worker's split is preserved because the ZIP supplied explicit splits; if you supplied a single `train.csv`, the worker drew a seeded holdout and the baseline below uses the worker's recorded row counts only as a sanity check.

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, log_loss, mean_absolute_error, mean_squared_error, r2_score
def baseline_for(frame):
    if frame is None:
        return None
    y = frame[TARGET_COLUMN]
    if IS_CLASSIFIER:
        classes = sorted(train_df[TARGET_COLUMN].astype(str).unique().tolist())
        majority = train_df[TARGET_COLUMN].astype(str).value_counts().idxmax()
        freq = float((train_df[TARGET_COLUMN].astype(str) == majority).mean())
        proba = np.full((len(frame), len(classes)), (1 - freq) / max(len(classes) - 1, 1)); proba[:, classes.index(majority)] = freq
        return {"accuracy": float(accuracy_score(y.astype(str), np.full(len(frame), majority))),
                "balancedAccuracy": float(balanced_accuracy_score(y.astype(str), np.full(len(frame), majority))),
                "logLoss": float(log_loss(y.astype(str), proba, labels=classes))}
    mean = float(train_df[TARGET_COLUMN].mean()); pred = np.full(len(frame), mean); yt = y.to_numpy(float)
    return {"mae": float(mean_absolute_error(yt, pred)), "rmse": float(np.sqrt(mean_squared_error(yt, pred))), "r2": float(r2_score(yt, pred))}
baseline = {"validation": baseline_for(val_df), "test": baseline_for(test_df)}
print("trivial baseline:", baseline)

## 5. Fresh-boundary reload, metric reproduction, and new-data inference

The deployable artifact is the pair `model.tabpfn_fit` (fitted estimator state, including the in-context training rows) + `model.ckpt` (foundation weights) described by `artifact_manifest.json`; the fitted archive alone is not a model. The next cell copies the artifact directory to a fresh location, checks every manifest digest, and reconstructs the estimator through `serving/load_artifact.py` — the same loader DIMER serving uses — **not** the in-memory object the worker held. It then recomputes the validation metric from the reloaded estimator and requires it to match the worker's recorded value within `1e-6`: that is the equivalence claim, stronger than "it loaded". Finally it scores new unlabelled rows; `prediction` is the raw point estimate; no interval column exists because the pipeline provides none. The `.tabpfn_fit` archive contains serialized Python/torch state — loading it executes trusted model state, so only load artifacts from a producer you trust.

In [ ]:
ART_SRC = OUTPUT_DIR / "artifacts"
manifest = json.loads((ART_SRC / "artifact_manifest.json").read_text(encoding="utf-8"))
RELOAD = Path("/content/tabpfn-artifact-reload")
if RELOAD.exists():
    shutil.rmtree(RELOAD)
shutil.copytree(ART_SRC, RELOAD)
for key in ("fittedEstimator", "foundationCheckpoint"):
    p = RELOAD / manifest[key]
    if hashlib.sha256(p.read_bytes()).hexdigest() != manifest[key + "Sha256"]:
        raise RuntimeError(f"digest mismatch for {manifest[key]}")
print("manifest:", {k: manifest[k] for k in ("schemaVersion", "taskType", "targetColumn", "fittedEstimator", "foundationCheckpoint")}, "| features:", len(manifest["featureColumns"]))
reloaded = load_dimer_tabpfn_artifact(RELOAD, device="cuda" if torch.cuda.is_available() else "cpu")
FEATURES = manifest["featureColumns"]
if val_df is not None:
    X_val, y_val = val_df[FEATURES], val_df[TARGET_COLUMN]
    if IS_CLASSIFIER:
        recomputed = float(accuracy_score(y_val.astype(str), np.asarray(reloaded.predict(X_val)).astype(str)))
        recorded = float(metrics["validation"]["accuracy"])
    else:
        recomputed = float(mean_absolute_error(y_val.to_numpy(float), np.asarray(reloaded.predict(X_val), dtype=float)))
        recorded = float(metrics["validation"]["mae"])
    if abs(recomputed - recorded) > 1e-6:
        raise RuntimeError(f"reloaded artifact does not reproduce the recorded validation metric: {recomputed} vs {recorded}")
    print(f"PASS: reloaded artifact reproduces the recorded validation metric ({recomputed:.6f} == {recorded:.6f} within 1e-6).")
new_rows = (test_df if test_df is not None else val_df)[FEATURES].head(8).copy()
pred = np.asarray(reloaded.predict(new_rows))
out = pd.DataFrame({"row_id": new_rows.index.to_numpy(), "prediction": pred})
if IS_CLASSIFIER:
    proba = np.asarray(reloaded.predict_proba(new_rows))
    for i, label in enumerate(manifest["classes"]):
        out[f"proba_{label}"] = proba[:, i]
print(out.head())

## 6. Machine-readable outputs and provenance

Predictions go to CSV with `row_id` mapping back to the input rows. Provenance merges the worker's own `evaluation/report.json` record (model generation, resolved base-model path, tabpfn/torch versions, dataset fingerprint, seed) with the tutorial's runtime, pins, data identity, baseline, and reload-equivalence result. It contains no credentials.

In [ ]:
OUT = Path("/content/tabpfn-tutorial-output")
OUT.mkdir(parents=True, exist_ok=True)
out.to_csv(OUT / "tabpfn_regression_predictions.csv", index=False)
report = json.loads((OUTPUT_DIR / "evaluation" / "report.json").read_text(encoding="utf-8"))
provenance = {
    "worker": {"repository": FT_REPO, "commit": FT_COMMIT, "runResult": result["message"], "mode": metrics.get("mode"),
               "fineTuneEffective": metrics.get("fineTuneEffective"), "fineTuneSkippedReason": metrics.get("fineTuneSkippedReason")},
    "model": {**report["provenance"]["model"], "modelVersionSelected": MODEL_VERSION},
    "runtime": {"python": platform.python_version(), "torch": md.version("torch"), "tabpfn": md.version("tabpfn"),
                "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"},
    "data": {"source": data_source, "sha256": dataset_digest, "target": TARGET_COLUMN, "splits": "explicit train/val/test from ZIP" if zips else f"worker seeded holdout {VALIDATION_SPLIT}",
             "rows": {"train": len(train_df), "val": None if val_df is None else len(val_df), "test": None if test_df is None else len(test_df)}},
    "metrics": {"worker": metrics, "trivialBaseline": baseline},
    "artifact": {k: manifest[k] for k in ("fittedEstimator", "fittedEstimatorSha256", "foundationCheckpoint", "foundationCheckpointSha256", "featureColumns")},
    "reloadEquivalence": {"recomputedValidationMetric": recomputed if val_df is not None else None, "tolerance": 1e-6},
    "seed": SEED,
}
(OUT / "tabpfn_regression_metrics.json").write_text(json.dumps({"worker": metrics, "trivialBaseline": baseline}, indent=2) + "\n")
(OUT / "tabpfn_regression_provenance.json").write_text(json.dumps(provenance, indent=2, default=str) + "\n")
print("Wrote prediction CSV plus metrics/provenance JSON to", OUT)

## Interpretation, limits, and next steps

A successful run proves the pinned worker revision installs and runs on this runtime exactly as DIMER invokes it, validates and fits in-context (or fine-tunes on a GPU) on the supplied data, records metrics that beat a trivial baseline on a synthetic sample, exports the `model.tabpfn_fit` + `model.ckpt` artifact pair with digests, and that the artifact reloaded from serialized files through the serving loader reproduces the recorded validation metric and scores new rows.

It does **not** establish benchmark quality, generalization, fairness, robustness, calibration, or production fitness — the sample is synthetic, the split is single, and no dispersion is estimated. A zero-shot fallback run is a different estimator from a fine-tuned run; compare them only with `fineTuneEffective` in view. Before any real deployment: use domain-appropriate splits, evaluate on representative data with subgroup breakdowns, estimate error intervals from held-out residuals, clear the selected generation's licence (`v3` is non-commercial), and record the clean-runtime release verification in `tutorials/README.md`.